# Chapter 2 — Memory Model: Virtual, Physical, IPA

## Concept
AArch64 4-level page tables (PGD→PUD→PMD→PTE), TTBR0_EL1 (user) /
TTBR1_EL1 (kernel), stage-1 and stage-2 translation for virtualisation,
Intermediate Physical Address (IPA) space, ASID tagging, and TLB maintenance.
QEMU `virt` machine places DRAM at **0x40000000** (1 GiB mark).

## Lab Objectives
1. Query guest physical memory layout via QMP (`query-memory-size-summary`).
2. Confirm RAM base address is **0x40000000** on the `virt` machine.
3. Hot-add 512 MiB via QMP object-add + device_add (`pc-dimm`).
4. Verify the new range appears in guest `/proc/iomem`.

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE_CODE = LABS_ROOT / "firmware" / "edk2-aarch64-code.fd"
FIRMWARE_VARS = LABS_ROOT / "firmware" / "varstore.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 1
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE_CODE.exists(), f"FATAL: Missing {FIRMWARE_CODE}. Re-run setup_qemu_labs.sh."
assert FIRMWARE_VARS.exists(), f"FATAL: Missing {FIRMWARE_VARS}. Re-run setup_qemu_labs.sh."
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware code       : {FIRMWARE_CODE}")
print(f"  Firmware vars       : {FIRMWARE_VARS}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    # Memory hotplug requires slots + maxmem at construction time.
    # 2 slots lets us hot-add 512 MiB in Step 3 and leave room for a second DIMM.
    maxmem="4G", memory_slots=2,
    firmware_code=str(FIRMWARE_CODE),
    firmware_vars=str(FIRMWARE_VARS),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=[],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

# -serial tcp::PORT,server,nowait means the guest boots whether or not a
# listener is attached. If this cell runs late relative to Cell 2, the
# login banner may have scrolled past before nc connected. Nudge getty
# to reprint — a no-op on a fresh boot.
sc.sendline("")
print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: Query base memory configuration ──────────────────────────────────
import json
mem = qmp.query_memory()
print("query-memory-size-summary:", json.dumps(mem, indent=2))
base_ram_bytes = mem.get("base-memory", 0)
plugged_bytes  = mem.get("plugged-memory", 0)
print(f"\nBase RAM   : {base_ram_bytes / (1024**3):.2f} GiB ({base_ram_bytes:#010x} bytes)")
print(f"Plugged RAM: {plugged_bytes  / (1024**2):.0f} MiB")


In [ ]:
# ── Step 2: Read /proc/iomem — confirm DRAM at 0x40000000 ────────────────────
iomem = sc.run_command("sudo cat /proc/iomem", timeout=15)
print(iomem[:2000])


In [ ]:
# ── Step 3: Hot-add 512 MiB via QMP (memory-backend-ram + pc-dimm) ───────────
# The launcher above configured memory_slots=2 and maxmem="4G", so these
# calls are expected to succeed. A failure here is a real regression, not
# a graceful-degradation case.
HOTPLUG_SIZE_MiB = 512
HOTPLUG_SIZE_B   = HOTPLUG_SIZE_MiB * 1024 * 1024

obj_resp = qmp.object_add(
    qom_type="memory-backend-ram",
    obj_id="hotmem0",
    size=HOTPLUG_SIZE_B,
)
print("object-add response:", obj_resp)

dev_resp = qmp.device_add(
    driver="pc-dimm",
    device_id="dimm0",
    memdev="hotmem0",
)
print("device_add response:", dev_resp)


In [ ]:
# ── Step 4: Confirm new range in /proc/iomem (if hot-plug succeeded) ─────────
time.sleep(3)   # Allow guest kernel to enumerate new device
iomem_after = sc.run_command("sudo cat /proc/iomem", timeout=15)
print(iomem_after[:3000])

mem_after = qmp.query_memory()
print("\nMemory after hot-plug:", json.dumps(mem_after, indent=2))


In [ ]:
# ── Step 5: Check guest memory zones (NUMA / zones) ──────────────────────────
zones = sc.run_command("cat /proc/zoneinfo | head -60", timeout=10)
print(zones)


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
RAM_SIZE_GiB = 2   # matches RAM="2G" in CONFIG

assert_true(base_ram_bytes > 0,
            "query-memory-size-summary returns base-memory > 0",
            detail=f"{base_ram_bytes / (1024**3):.2f} GiB",
            action="Check -m parameter in QEMULauncher")

assert_in_range(base_ram_bytes / (1024**3),
                RAM_SIZE_GiB * 0.9, RAM_SIZE_GiB * 1.1,
                f"Base RAM is ~{RAM_SIZE_GiB} GiB",
                unit=" GiB",
                action="Check RAM= in CONFIG block")

# 0x40000000 = 1073741824 = 1 GiB — DRAM start on virt machine
assert_contains(iomem, r"40000000",
                "DRAM starts at 0x40000000 in /proc/iomem",
                action="Check -machine virt; x86 base addresses differ")

assert_contains(iomem, r"System RAM",
                "/proc/iomem contains 'System RAM' entry",
                action="Guest kernel did not map DRAM in iomem")

assert_contains(iomem_after, r"40000000",
                "Hot-added RAM appears in /proc/iomem",
                action="Guest kernel did not detect hot-plugged DIMM")

assert_true(
    mem_after.get("plugged-memory", 0) >= HOTPLUG_SIZE_B * 0.9,
    "query-memory-size-summary reflects hot-plugged memory",
    detail=f"plugged={mem_after.get('plugged-memory',0)/(1024**2):.0f} MiB",
    action="QMP plugged-memory not updated — check pc-dimm attachment",
)


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| base-memory > 0 | Yes | QMP reports configured RAM |
| Base RAM ≈ 2 GiB | 2 GiB ± 10% | Matches -m 2G |
| DRAM at 0x40000000 | Present | virt machine fixed DRAM base |
| System RAM in /proc/iomem | Present | Guest kernel mapped DRAM |
| Hot-add 512 MiB | Success | Requires -machine maxmem |
| plugged-memory updated | ≥ 512 MiB | QMP reflects hot-plug |

> **Fidelity gap**: Stage-2 IPA translation is functionally emulated.
> Performance characteristics (TLB hit rates, walk latency) differ from
> real Neoverse N2 with two-stage translation.

## Next Steps
→ **Chapter 3**: Exception Levels — observe EL transitions in the serial boot log.
